# Multi-Robot Path Planning using A* Algorithms

This notebook implements:
- **Phase 1**: Independent A* (baseline)
- **Phase 2**: Conflict Detection System
- **Phase 3**: Cooperative A* with reservations
- **Phase 4**: Testing & Validation
- **Phase 5**: Integration wrapper

In [1]:
# ========================================
# IMPORTS
# ========================================
import heapq
import time
from collections import defaultdict
from typing import List, Dict, Tuple, Optional, Set

In [2]:
# ========================================
# NODE CLASS (from Issam_Laribi)
# ========================================
class Node:
    """Represents a node in the search tree"""

    def __init__(self, position, parent=None, g=0, h=0):
        """Initialize a node for pathfinding
        
        Args:
            position (tuple): (x, y) coordinate
            parent (Node): Parent node in the path
            g (float): Cost from start to this node
            h (float): Heuristic cost from this node to goal (Manhattan distance)
        """
        self.position = position
        self.parent = parent
        self.g = g
        self.h = h
        self.f = g + h

    def __lt__(self, other):
        """Compare nodes for priority queue (min-heap by f value)"""
        return self.f < other.f

    def __eq__(self, other):
        """Check if two nodes represent the same position"""
        return self.position == other.position

    def __hash__(self):
        """Hash function for use in sets and dicts"""
        return hash(self.position)

    def reconstruct_path(self):
        """Reconstruct path from start to this node
        
        Returns:
            List[tuple]: Path as list of (x, y) coordinates
        """
        path = []
        current = self
        while current is not None:
            path.append(current.position)
            current = current.parent
        return list(reversed(path))

In [3]:
# ========================================
# GRID ENVIRONMENT (from Issam_Laribi)
# ========================================
class GridEnvironment:
    """Represents the grid map"""

    def __init__(self, filename=None):
        """Load grid from file or create empty grid"""
        self.grid = []
        self.height = 0
        self.width = 0
        
        if filename:
            self.load_from_file(filename)

    def load_from_file(self, filename):
        """Load grid from file"""
        self.grid = []
        with open(filename, 'r') as f:
            for line in f:
                row = []
                for char in line.strip():
                    if char == '.':
                        row.append(True)
                    elif char == 'T':
                        row.append(False)
                self.grid.append(row)
        self.height = len(self.grid)
        self.width = len(self.grid[0]) if self.height > 0 else 0

    def is_valid_position(self, x, y):
        """Check if position is within grid bounds"""
        return 0 <= x < self.width and 0 <= y < self.height

    def is_walkable(self, x, y):
        """Check if position is walkable (not obstacle)"""
        return self.is_valid_position(x, y) and self.grid[y][x]

    def get_neighbors(self, x, y):
        """Get adjacent cells (4-directional)"""
        directions = [(0, 1), (1, 0), (0, -1), (-1, 0)]
        neighbors = []

        for dx, dy in directions:
            nx, ny = x + dx, y + dy
            if self.is_walkable(nx, ny):
                neighbors.append((nx, ny))

        return neighbors

    def manhattan_distance(self, pos1, pos2):
        """Calculate Manhattan distance between two positions"""
        return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])

In [4]:
# ========================================
# ROBOT CLASS (from Issam_Laribi)
# ========================================
class Robot:
    """Represents a single robot in the warehouse"""
    def __init__(self, robot_id, grid, start_pos, goal_pos, color='blue'):
        """Initialize robot with grid reference
        
        Args:
            robot_id (int): Unique identifier for robot
            grid (GridEnvironment): The grid this robot moves in
            start_pos (tuple): Starting (x, y) position
            goal_pos (tuple): Goal (x, y) position
            color (str): Robot color for visualization
        """
        self.robot_id = robot_id
        self.grid = grid
        self.start_pos = start_pos
        self.goal_pos = goal_pos
        self.current_pos = start_pos
        self.path = []
        self.path_index = 0
        self.color = color

    def get_current_position(self):
        """Get current position"""
        return self.current_pos

    def get_goal_position(self):
        """Get goal position"""
        return self.goal_pos

    def get_start_position(self):
        """Get start position"""
        return self.start_pos

    def set_path(self, path):
        """Store path for robot"""
        self.path = path
        self.path_index = 0

    def get_path(self):
        """Get stored path"""
        return self.path

    def get_position_at_time(self, time_step):
        """Get robot's position at a specific time step"""
        if time_step < len(self.path):
            return self.path[time_step]
        else:
            return self.path[-1] if self.path else self.current_pos

    def is_at_goal(self):
        """Check if robot reached goal"""
        return self.current_pos == self.goal_pos

In [6]:
# ========================================
# PHASE 1: INDEPENDENT A* PLANNER
# ========================================

class IndependentAStarPlanner:
    """
    Independent A* planner - each robot is planned independently without awareness of others.
    This serves as the baseline for comparison.
    """

    def __init__(self, grid: GridEnvironment):
        self.grid = grid

    def run_independent_astar(self, robot: Robot, obstacles: Set = None) -> List[Tuple[int, int]]:
        """Run A* for a single robot."""
        if obstacles is None:
            obstacles = set()

        start = robot.start_pos
        goal = robot.goal_pos

        if not self.grid.is_walkable(*start) or not self.grid.is_walkable(*goal):
            return None

        if start == goal:
            return [start]

        open_set = []
        start_node = Node(start, g=0, h=self.grid.manhattan_distance(start, goal))
        heapq.heappush(open_set, start_node)

        came_from = {}
        g_score = {start: 0}
        closed_set = set()

        while open_set:
            current = heapq.heappop(open_set)

            if current.position in closed_set:
                continue
            closed_set.add(current.position)

            if current.position == goal:
                return current.reconstruct_path()

            neighbors = self.grid.get_neighbors(*current.position)
            for neighbor in neighbors:
                if neighbor in obstacles:
                    continue

                tentative_g = current.g + 1

                if neighbor not in g_score or tentative_g < g_score[neighbor]:
                    g_score[neighbor] = tentative_g
                    h = self.grid.manhattan_distance(neighbor, goal)
                    neighbor_node = Node(neighbor, parent=current, g=tentative_g, h=h)
                    came_from[neighbor_node] = current
                    heapq.heappush(open_set, neighbor_node)

        return None

    def plan_all_robots(self, robots: List[Robot]) -> Dict[int, List[Tuple[int, int]]]:
        """Plan paths for all robots independently."""
        paths = {}
        for robot in robots:
            path = self.run_independent_astar(robot)
            paths[robot.robot_id] = path
        return paths

In [8]:
# ========================================
# PHASE 2: CONFLICT DETECTION SYSTEM
# ========================================

class ConflictDetector:
    """
    Conflict detection system that identifies three types of collisions between robot paths.
    
    Conflict Types:
    1. Vertex Conflict: Two robots occupy same position at same time step
    2. Edge Conflict: Two robots traverse same edge in same direction    """

    def __init__(self):b
    self.conflicts = []

    def detect_vertex_conflicts(self, paths: Dict) -> List[Dict]:
        """Detect vertex conflicts - two robots at same position at same time."""
        conflicts = []
        robot_ids = list(paths.keys())
        max_time = max(len(path) for path in paths.values())

        for t in range(max_time):
            positions_at_t = {}
            for robot_id in robot_ids:
                path = paths[robot_id]
                if path and t < len(path):
                    positions_at_t[robot_id] = path[t]

            position_to_robots = defaultdict(list)
            for robot_id, pos in positions_at_t.items():
                position_to_robots[pos].append(robot_id)

            for pos, robots in position_to_robots.items():
                if len(robots) > 1:
                    conflicts.append({
                        'type': 'vertex',
                        'robot1': robots[0],
                        'robot2': robots[1],
                        'time': t,
                        'position': pos
                    })
        return conflicts

    def detect_edge_conflicts(self, paths: Dict) -> List[Dict]:
        """Detect edge conflicts - two robots traverse same edge in same direction."""
        conflicts = []
        robot_ids = list(paths.keys())
        max_time = max(len(path) for path in paths.values()) - 1

        for t in range(max_time):
            edges_at_t = {}
            for robot_id in robot_ids:
                path = paths[robot_id]
                if path and t + 1 < len(path):
                    edge = (path[t], path[t + 1])
                    edges_at_t[(edge, robot_id)] = edge

            edge_to_robots = defaultdict(list)
            for (edge, robot_id), edge_pos in edges_at_t.items():
                edge_to_robots[edge_pos].append(robot_id)

            for edge, robots in edge_to_robots.items():
                if len(robots) > 1:
                    conflicts.append({
                        'type': 'edge',
                        'robot1': robots[0],
                        'robot2': robots[1],
                        'time': t,
                        'edge': edge
                    })
        return conflicts

    def get_conflict_report(self, paths: Dict) -> List[Dict]:
        """Get complete conflict report."""
        vertex_conflicts = self.detect_vertex_conflicts(paths)
        edge_conflicts = self.detect_edge_conflicts(paths)
        self.conflicts = vertex_conflicts + edge_conflicts
        return self.conflicts

    def count_conflicts(self, paths: Dict) -> int:
        """Count total number of conflicts."""
        return len(self.get_conflict_report(paths))

    def pad_paths(self, paths: Dict) -> Dict:
        """Pad shorter paths with wait actions at goal."""
        max_time = max(len(path) for path in paths.values()) if paths else 0
        padded_paths = {}

        for robot_id, path in paths.items():
            if path and len(path) > 0:
                goal_pos = path[-1]
                padded_path = path + [goal_pos] * (max_time - len(path))
                padded_paths[robot_id] = padded_path
        return padded_paths

    def build_occupation_map(self, paths: Dict) -> Dict[Tuple, str]:
        """Build time-space occupation map."""
        occupation_map = {}
        padded_paths = self.pad_paths(paths)

        for robot_id, path in padded_paths.items():
            for t, position in enumerate(path):
                occupation_map[(position[0], position[1], t)] = robot_id
        return occupation_map

NameError: name 'self' is not defined

In [ ]:
# ========================================
# PHASE 3: COOPERATIVE A* PLANNER
# ========================================

class CooperativeAStarPlanner:
    """
    Cooperative A* planner - sequential planning where each robot considers
    previously planned robots' paths as obstacles with penalty.
    """

    def __init__(self, grid: GridEnvironment, penalty: int = 1000):
        self.grid = grid
        self.penalty = penalty
        self.RESERVATION_PENALTY = penalty

    def get_reserved_positions(self, planned_paths: Dict) -> Dict[Tuple, str]:
        """Build time-space reservation map from planned paths."""
        if not planned_paths:
            return {}

        max_time = max(len(path) for path in planned_paths.values())
        reserved = {}

        for robot_id, path in planned_paths.items():
            for t, position in enumerate(path):
                reserved[(position[0], position[1], t)] = robot_id

            goal_pos = path[-1]
            for t in range(len(path), max_time):
                reserved[(goal_pos[0], goal_pos[1], t)] = robot_id

        return reserved

    def a_star_with_reservations(self, robot: Robot, reserved_positions: Dict = None) -> List[Tuple[int, int]]:
        """Modified A* that considers reserved positions with penalty."""
        if reserved_positions is None:
            reserved_positions = {}

        start = robot.start_pos
        goal = robot.goal_pos

        if not self.grid.is_walkable(*start) or not self.grid.is_walkable(*goal):
            return None

        if start == goal:
            return [start]

        open_set = []
        start_node = Node(start, g=0, h=self.grid.manhattan_distance(start, goal))
        heapq.heappush(open_set, start_node)

        came_from = {}
        g_score = {(start, 0): 0}
        closed_set = set()

        while open_set:
            current = heapq.heappop(open_set)

            state = (current.position, current.g)
            if state in closed_set:
                continue
            closed_set.add(state)

            if current.position == goal:
                return current.reconstruct_path()

            neighbors = self.grid.get_neighbors(*current.position)
            time_step = current.g

            for neighbor in neighbors:
                time_step = current.g + 1
                
                tentative_g = current.g + 1

                if (neighbor[0], neighbor[1], time_step) in reserved_positions:
                    tentative_g += self.RESERVATION_PENALTY

                new_state = (neighbor, tentative_g)
                if new_state not in g_score or tentative_g < g_score[new_state]:
                    g_score[new_state] = tentative_g
                    h = self.grid.manhattan_distance(neighbor, goal)
                    neighbor_node = Node(neighbor, parent=current, g=tentative_g, h=h)
                    came_from[neighbor_node] = current
                    heapq.heappush(open_set, neighbor_node)

        return None

    def sort_robots(self, robots: List[Robot], order: str = 'fixed') -> List[Robot]:
        """Sort robots by planning order."""
        if order == 'fixed':
            return robots
        elif order == 'path_length':
            return sorted(robots, key=lambda r: self.grid.manhattan_distance(r.start_pos, r.goal_pos))
        return robots

    def plan_robots_sequentially(self, robots: List[Robot], ordering: str = 'fixed') -> Dict[int, List[Tuple[int, int]]:
        """Plan robots sequentially, considering previously planned paths."""
        sorted_robots = self.sort_robots(robots, ordering)
        planned_paths = {}

        for robot in sorted_robots:
            reserved = self.get_reserved_positions(planned_paths)
            path = self.a_star_with_reservations(robot, reserved)

            if path is None:
                raise PlanningFailure(f"No path found for robot {robot.robot_id}")

            planned_paths[robot.robot_id] = path
        return planned_paths

    def retry_with_higher_penalty(self, robot: Robot, reserved: Dict, multiplier: float = 2.0) -> List[Tuple[int, int]]:
        """Retry planning with higher penalty."""
        old_penalty = self.RESERVATION_PENALTY
        self.RESERVATION_PENALTY = int(old_penalty * multiplier)

        try:
            return self.a_star_with_reservations(robot, reserved)
        finally:
            self.RESERVATION_PENALTY = old_penalty


class PlanningFailure(Exception):
    """Exception raised when path planning fails."""
    pass

In [ ]:
# ========================================
# PHASE 4: TESTING & VALIDATION
# ========================================

def validate_paths(paths: Dict, grid: GridEnvironment, robots: List[Robot]) -> Tuple[bool, List[str]]:
    """Validate that all paths are valid."""
    errors = []

    for robot in robots:
        path = paths.get(robot.robot_id)

        if path is None or len(path) == 0:
            errors.append(f"Robot {robot.robot_id}: No path found")
            continue

        if path[0] != robot.start_pos:
            errors.append(f"Robot {robot.robot_id}: Path does not start at start position")

        if path[-1] != robot.goal_pos:
            errors.append(f"Robot {robot.robot_id}: Path does not end at goal position")

        for i, pos in enumerate(path):
            if not grid.is_walkable(*pos):
                errors.append(f"Robot {robot.robot_id}: Invalid position {pos} at step {i}")

        for i in range(len(path) - 1):
            dx = abs(path[i+1][0] - path[i][0])
            dy = abs(path[i+1][1] - path[i][1])
            if dx + dy != 1 and dx + dy != 0:
                errors.append(f"Robot {robot.robot_id}: Invalid move from {path[i]} to {path[i+1]}")

    return len(errors) == 0, errors


def count_conflicts(paths: Dict) -> int:
    """Count total number of conflicts."""
    detector = ConflictDetector()
    return detector.count_conflicts(paths)


def calculate_makespan(paths: Dict) -> int:
    """Calculate makespan - maximum path length."""
    return max(len(path) for path in paths.values()) if paths else 0


def calculate_flowtime(paths: Dict) -> int:
    """Calculate flowtime - sum of all path lengths."""
    return sum(len(path) for path in paths.values()) if paths else 0


def measure_plan_time(func) -> Tuple[float, any]:
    """Measure execution time of a function."""
    start_time = time.time()
    result = func()
    elapsed_time = time.time() - start_time
    return elapsed_time, result


def compare_algorithms(grid: GridEnvironment, robots: List[Robot]) -> Dict:
    """Run both Independent and Cooperative algorithms and compare results."""
    independent_planner = IndependentAStarPlanner(grid)
    cooperative_planner = CooperativeAStarPlanner(grid)

    ind_time, ind_paths = measure_plan_time(lambda: independent_planner.plan_all_robots(robots))
    coop_time, coop_paths = measure_plan_time(lambda: cooperative_planner.plan_robots_sequentially(robots))

    ind_valid, ind_errors = validate_paths(ind_paths, grid, robots)
    coop_valid, coop_errors = validate_paths(coop_paths, grid, robots)

    results = {
        'independent': {
            'paths': ind_paths,
            'valid': ind_valid,
            'errors': ind_errors,
            'conflicts': count_conflicts(ind_paths),
            'makespan': calculate_makespan(ind_paths),
            'flowtime': calculate_flowtime(ind_paths),
            'plan_time': ind_time
        },
        'cooperative': {
            'paths': coop_paths,
            'valid': coop_valid,
            'errors': coop_errors,
            'conflicts': count_conflicts(coop_paths),
            'makespan': calculate_makespan(coop_paths),
            'flowtime': calculate_flowtime(coop_paths),
            'plan_time': coop_time
        }
    }

    return results

In [ ]:
# ========================================
# PHASE 5: INTEGRATION WRAPPER
# ========================================

class MultiRobotPlanner:
    """
    Unified multi-robot planner that selects between Independent and Cooperative planning.
    """

    def __init__(self, grid: GridEnvironment, algorithm: str = 'independent', penalty: int = 1000):
        self.grid = grid
        self.algorithm = algorithm
        self.penalty = penalty

        if algorithm == 'cooperative':
            self.planner = CooperativeAStarPlanner(grid, penalty)
        else:
            self.planner = IndependentAStarPlanner(grid)

    def choose_algorithm(self, algorithm_type: str):
        """Select planning algorithm."""
        self.algorithm = algorithm_type
        if algorithm_type == 'cooperative':
            self.planner = CooperativeAStarPlanner(self.grid, self.penalty)
        else:
            self.planner = IndependentAStarPlanner(self.grid)

    def execute(self, robots: List[Robot], ordering: str = 'fixed') -> Dict[int, List[Tuple[int, int]]]:
        """Execute the selected planning algorithm."""
        if self.algorithm == 'cooperative':
            return self.planner.plan_robots_sequentially(robots, ordering)
        else:
            return self.planner.plan_all_robots(robots)

    def get_results_summary(self, paths: Dict, robots: List[Robot]) -> Dict:
        """Get summary of results."""
        valid, errors = validate_paths(paths, self.grid, robots)
        detector = ConflictDetector()
        conflicts = detector.get_conflict_report(paths)

        return {
            'algorithm': self.algorithm,
            'paths': paths,
            'valid': valid,
            'errors': errors,
            'conflicts': conflicts,
            'conflict_count': len(conflicts),
            'makespan': calculate_makespan(paths),
            'flowtime': calculate_flowtime(paths)
        }